# Missouri -- Title XXIV Chapters 374–385 (Insurance) -> `data/missouri/ins_codes/*.md`

Missouri **insurance** statutes live under **Title XXIV – Business and Financial Institutions** as **Chapters 374–385** (official: [revisor.mo.gov](https://revisor.mo.gov/main/OneChapter.aspx?chapter=375)). Earlier Title XXIV chapters (361–373) cover banks and credit institutions; this notebook **only** crawls **374–385**.

Chapter pages also link to **other publication years** (2023, 2025, etc.). Discovery and downloads **only** follow paths under **`/codes/missouri/{CODE_YEAR}/`** so the corpus stays one year. Collected section URLs are **normalized** to that year.

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`** (same pattern as **`ins_ipynb/mississippi.ipynb`**).

**Discovery:** BFS from each **Chapter 374–385** index URL under **`/codes/missouri/{CODE_YEAR}/title-xxiv/`**; every **`section-<chapter>-...`** link whose chapter is in **374–385** is collected (~**1,500** sections for **2024**).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`MO_sec_<label>.md`** where **`<label>`** is the segment after **`section-`** (hyphens to underscores).

**Config:** **`CODE_YEAR`** seeds chapter index URLs. **`MAX_SECTIONS`** caps downloads (**0** = all). **`MAX_DISCOVERY_PAGES`** caps index fetches (**0** = no cap). **`REUSE_DISCOVERED_URLS`** skips discovery when **`_missouri_ins_ch374_385_section_urls.txt`** exists.

Run with the **project root** as cwd (same as **`python -m app.ingest`**).

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from collections import deque
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
CODE_YEAR = "2024"
PATH_PREFIX = "/codes/missouri/title-xxiv"
ALLOWED_CHAPTERS = frozenset(range(374, 386))  # 374–385 inclusive

CHAPTER_INDEXES = [
    f"{BASE}/codes/missouri/{CODE_YEAR}/title-xxiv/chapter-{ch}/"
    for ch in sorted(ALLOWED_CHAPTERS)
]

OUT_DIR = Path("data") / "missouri" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_missouri_ins_ch374_385_section_urls.txt"
REUSE_DISCOVERED_URLS = True

section_label_re = re.compile(r"/section-([^/]+)/?$", re.I)
chapter_from_path_re = re.compile(r"/chapter-(\d+)(?:/|$)")


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def canonical(p: str) -> str:
    """Map /codes/missouri/YYYY/... to /codes/missouri/... (used only for chapter parsing)."""
    return re.sub(r"^(/codes/missouri)/\d{4}/", r"\1/", p)


def chapter_from_path(p: str) -> int | None:
    c = canonical(p)
    m = chapter_from_path_re.search(c + "/")
    return int(m.group(1)) if m else None


def in_scope(p: str) -> bool:
    if f"/codes/missouri/{CODE_YEAR}/" not in p:
        return False
    c = canonical(p)
    if c != PATH_PREFIX and not c.startswith(PATH_PREFIX + "/"):
        return False
    ch = chapter_from_path(c)
    if ch is None:
        return False
    return ch in ALLOWED_CHAPTERS


def normalized_section_url_from_path(p: str) -> str:
    m = section_label_re.search(p)
    if not m:
        raise ValueError(f"not a section path: {p!r}")
    label = m.group(1)
    head = label.split("-")[0]
    if not head.isdigit():
        raise ValueError(f"bad section label: {label!r}")
    ch = int(head)
    return f"{BASE}/codes/missouri/{CODE_YEAR}/title-xxiv/chapter-{ch}/section-{label}/"


def is_section_path(p: str) -> bool:
    m = section_label_re.search(p)
    if not m:
        return False
    label = m.group(1)
    head = label.split("-")[0]
    if not head.isdigit():
        return False
    return int(head) in ALLOWED_CHAPTERS


def discover_section_urls() -> list[str]:
    """BFS chapter index pages for 374–385; collect section URLs."""
    seen: set[str] = set()
    in_q: set[str] = set()
    q: deque[str] = deque()
    for start in CHAPTER_INDEXES:
        pk = path_key(start)
        if pk not in seen and pk not in in_q:
            in_q.add(pk)
            q.append(start)

    sections: set[str] = set()
    fetches = 0

    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url)
        in_q.discard(pk)
        if pk in seen:
            continue
        if is_section_path(pk):
            continue
        if not in_scope(pk):
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"]).split("#")[0]
            p = path_key(absu)
            if not in_scope(p):
                continue
            if is_section_path(p):
                sections.add(normalized_section_url_from_path(p))
            else:
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")

    return sorted(sections, key=lambda u: label_sort_key(statute_label_from_url(u)))


def statute_label_from_url(url: str) -> str:
    pk = path_key(url)
    m = section_label_re.search(pk)
    if not m:
        raise ValueError(f"cannot parse section from {url!r}")
    return m.group(1)


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_filename(label: str) -> str:
    safe = label.replace("-", "_")
    return f"MO_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
        "View Our Newest Version Here",
    )
    drop_exact = {
        "of",
        "this Section",
        "Universal Citation:",
        "Next",
        "Previous",
    }
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if s in drop_exact:
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and "Missouri" in s and "Stat" in s:
            continue
        if s.startswith("There Is a Newer Version"):
            continue
        if s.startswith("Disclaimer:") or s.startswith("These codes may not"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_missouri_insurance() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(statute_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs (Title XXIV, chapters 374–385)")
        all_urls = found
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = statute_label_from_url(sec_url)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t.split("::", 1)[0].strip() if head_t else f"Missouri Revised Statutes § {label.replace('-', '.')}"
                md = (
                    f"# {title}\n\n"
                    f"**Missouri Revised Statutes -- Title XXIV, Chapters 374–385 (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [revisor.mo.gov](https://revisor.mo.gov/main/OneChapter.aspx?chapter=375)\n\n"
                    f"**Section (URL label):** {label}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_missouri_insurance()


Discovered 1539 section URLs (Title XXIV, chapters 374–385)
… 200/1539 (wrote=200 skipped=0 failed=0)
… 400/1539 (wrote=400 skipped=0 failed=0)
… 600/1539 (wrote=600 skipped=0 failed=0)
… 800/1539 (wrote=800 skipped=0 failed=0)
… 1000/1539 (wrote=1000 skipped=0 failed=0)
… 1200/1539 (wrote=1200 skipped=0 failed=0)
… 1400/1539 (wrote=1400 skipped=0 failed=0)
Done. wrote=1539 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/missouri/ins_codes


{'wrote': 1539, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
